# Evaluate Term Dispersion Scores on the GENIA Corpus Data and Reproduce Results Reported in the Mannuscript 

Authors: Samuel Sarria Hurtado, Uyen "Rachel" Lai, and Paul Sheridan

Description: Evaluate the following term dispersion scores on the Genia corpus data:
- Inverse Document Frequency (IDF)
- Inverse Collection Frequency (ICF)
- Chi-square
- Church and Gale (CG)
- Irvine and Callison-Burch (ICB)
- Derivation of Proportions (DoP)
- Residual ICF (RICF)

Calculate average P@k scores for each scoring function using the GENIA terms as ground truth. Also, evaluate scoring functions for their ability to filter out stopwords. 

## Preliminaries

In [1]:
# Imports
import sys
import os
import pickle
import json
import pandas as pd
sys.path.append('../../')
import wordstats
from sklearn.feature_extraction.text import CountVectorizer
import random
import numpy as np
import scipy
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
from io import StringIO
from numpy import nan
from tqdm import tqdm
import rbo

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/pasheridan/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Load the GENIA Corpus Data

In particular, we load the preprocessed GENIA corpus documents, and gold standard biological terms (i.e., lexical units) and their associated semantic classes (i.e., sems) and associated high-level class (i.e., amino_acid, nucleotide, multi_cell, cell, and other).

First, load the corpus docs, and the lexical units. Then hardcode the high-level semantic classes.

In [2]:
# Load the preprocessed GENIA corpus documents
genia_corpus_path = '../1-preprocessing/GENIAcorpus3.02-preprocessed.json'

with open(genia_corpus_path, "r") as j:
  genia_corpus = json.loads(j.read())

# Load gold standard terms 
genia_keywords_path = '../1-preprocessing/GENIAcorpus3.02-keywords.tsv'

with open(genia_keywords_path, "r") as c:
  genia_lexical_units_and_sems = pd.read_csv(c, sep='\t')

genia_lexical_units = genia_lexical_units_and_sems.lex.to_numpy()

# Hardcode the low-level semantic classes and their associated high-level abstract semantic classes
amino_acid_sems = ['G#amino_acid_monomer', 'G#peptide', 'G#protein_N/A',
              'G#protein_complex', 'G#protein_domain_or_region',
              'G#protein_family_or_group', 'G#protein_molecule',
              'G#protein_substructure', 'G#protein_subunit',
              'G#other_organic_compound', 'G#organic', 'G#inorganic', 'G#atom',
              'G#carbohydrate', 'G#lipid']
nucleotide_sems = ['G#nucleotide', 'G#polynucleotide', 'G#DNA_N/A',
        'G#DNA_domain_or_region', 'G#DNA_family_or_group', 'G#DNA_molecule',
        'G#DNA_substructure', 'G#RNA_N/A', 'G#RNA_domain_or_region',
        'G#RNA_family_or_group', 'G#RNA_molecule', 'G#RNA_substructure']
multi_cell_sems = ['G#virus', 'G#mono_cell', 'G#multi_cell', 'G#body_part', 'G#tissue']
cell_sems = ['G#cell_type', 'G#cell_component', 'G#cell_line', 'G#other_artificial_source']
other_sems = ['G#other_name']
high_level_semantic_class_names = ['amino_acid', 'nucleotide', 'multi_cell', 'cell', 'other']
high_level_semantic_class_lex_units = [genia_lexical_units, amino_acid_sems, nucleotide_sems, multi_cell_sems, cell_sems, other_sems]

Process high-level semantic classes.

In [3]:
# Collect lexical units belonging to a given high-level semantic class
def get_high_level_semantic_class_words(high_level_class_lst):
  words = []
  for k, v in lex_sem_dct.items():
    if v in high_level_class_lst:
      words.append(k)
  return words

# Create dictionary of lexical units and their associated semantic classes
sem = np.array(genia_lexical_units_and_sems['sem'])
lex = np.array(genia_lexical_units_and_sems['lex'])
lex_sem_dct = dict(zip(lex, sem))

# Create data frame of lexical units, semantic classes, and high-level semantic classes
lex_size = len(genia_lexical_units_and_sems) # Number of lexical units in the vocabulary
high_level_sems_lst = [] # Initialize list for recording high-level semantic classes

# For each term in the vocab, identify low-level semantic class with high-level one 
for index in range(lex_size):
    low_level_sem = genia_lexical_units_and_sems.iloc[index, 1]
    if low_level_sem in amino_acid_sems:
        high_level_sems_lst.append('amino_acid')
    elif low_level_sem in nucleotide_sems:
        high_level_sems_lst.append('nucleotide')
    elif low_level_sem in multi_cell_sems:
        high_level_sems_lst.append('multi_cell')
    elif low_level_sem in cell_sems:
        high_level_sems_lst.append('cell')
    else:
        high_level_sems_lst.append('other')

# Add high-level semantic classes to data frame
genia_lexical_units_and_sems['class'] = high_level_sems_lst

# Print to console:
display(genia_lexical_units_and_sems)

,lex,sem,class
0,IL-2_gene_expression_lex,G#other_name,other
1,IL-2_gene_lex,G#DNA_domain_or_region,nucleotide
2,NF-kappa_B_activation_lex,G#other_name,other
3,NF-kappa_B_lex,G#protein_molecule,amino_acid
4,CD28_lex,G#protein_molecule,amino_acid
...,...,...,...
31782,gp160-induced_AP-1_complex_lex,G#protein_complex,amino_acid
31783,protein_synthesis-independent_lex,G#other_name,other
31784,calcium_channel_blocker_lex,G#other_organic_compound,amino_acid
31785,anti-CD3-induced_interleukin-2_secretion_lex,G#other_name,other


## Prepare the GENIA Corpus Data for Analysis

Prepare the corpus vocabulary.

In [4]:
# Compile the GENIA corpus vocabulary
pre_vocab = []
for i in range(len(genia_corpus)):
  pre_vocab.append(genia_corpus[i].split())

vocab = []
for i in range(len(pre_vocab)):
  for j in range(len(pre_vocab[i])):
    vocab.append(pre_vocab[i][j])

vocab = list(set(vocab))
vocab.sort()

# Helper function to ensure that CountVectorizer doesn't ignore any terms
def analyzer_custom(doc):
  return doc.split()

# Convert GENIA documents into term-in-document matrix of token counts.
counter = CountVectorizer(lowercase=False, vocabulary=vocab, analyzer=analyzer_custom)
collection = counter.transform(genia_corpus)

## Evaluate Term Dispersion Scores for Selected Measures

Calculate bag-of-words model word statistics and related quantities.

In [5]:
# Calculate word statistics and related quantities
m = len(counter.get_feature_names_out()) # vocab size
d = collection.shape[0] # collection size
N_i = wordstats.get_Ni(collection)
N_j = wordstats.get_Nj(collection)
N = wordstats.get_N(N_j)
B_ij = wordstats.get_Bij(collection)
B_i = wordstats.get_Bi(B_ij)
B_j = wordstats.get_Bj(B_ij)
DF = wordstats.get_DF(B_i, d)
CF = wordstats.get_CF(N_i)
nij_by_nj = wordstats.get_nij_by_nj(collection, N_j)
thetas = np.array(range(1, max(N_i.A[0]) + 1))/N
opt_thetas = wordstats.get_opt_thetas(N, m, d, N_i, N_j, B_i, thetas)

Evaluate term dispersion scores.

In [6]:
# Calculate word dispersion scores according the various measures used in this study
IDF = wordstats.get_IDF(DF)
ICF = wordstats.get_ICF(CF)
Chisq = wordstats.get_Chisq(collection)
CG = wordstats.get_CG(N_i, B_i)
ICB = wordstats.get_ICB(nij_by_nj, B_i)
DoP = wordstats.get_DoP(collection, N_i, N_j, N)
RICF = wordstats.get_RICF(opt_thetas, N, ICF)

/Users/pasheridan/Desktop/github-repos/bursty-term-measure/genia/3-tables/../../wordstats.py:209: RuntimeWarning: divide by zero encountered in log
  return -np.log(chisq_values)


Arrange term dispersion scores into a data frame.

In [7]:
# Initialize term dispersion scores data frame (augmented with ni and bi values)
term_scores_aug_df = pd.DataFrame(data=
                    {'lex': counter.get_feature_names_out(),
                     'IDF': IDF.A[0],
                     'ICF': ICF.A[0],
                     'Chi-sq': Chisq,
                     'CG': CG.A[0],
                     'ICB': ICB.A[0],
                     'DoP': DoP.A[0],
                     'RICF': RICF.A[0],
                     'bi': B_i.A[0],
                     'ni': N_i.A[0]})

# Augment with low-level and high-level semantic classes
term_scores_aug_df = pd.merge(term_scores_aug_df, genia_lexical_units_and_sems, on='lex', how='left')

# Tidy up the data frame
new_order = ['lex', 'sem', 'class', 'bi', 'ni', 'IDF', 'ICF', 'Chi-sq', 'CG', 'ICB', 'DoP', 'RICF'] # Define column ordering
term_scores_aug_df = term_scores_aug_df.reindex(columns=new_order)
term_scores_aug_df = term_scores_aug_df.rename(columns={'lex': 'term'}) # Rename 'lex' column to 'term'
all_duplicates = term_scores_aug_df[term_scores_aug_df.duplicated(keep=False)] # There are a few duplicate rows for some unknown reason
print("Duplicate rows:")
display(all_duplicates)
term_scores_aug_df = term_scores_aug_df.drop_duplicates() # Drop any duplicate rows

# Print to console
print("Term dispersion scores:")
display(term_scores_aug_df)

# Write to TSV
term_scores_aug_df.to_csv('term-dispersion-scores.tsv', sep='\t')

Duplicate rows:


,term,sem,class,bi,ni,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
16653,basic_helix-loop-helix_protein_lex,G#protein_family_or_group,amino_acid,4,6,6.214608,11.009194,310.072539,1.50,233.5,-0.001611,0.404383
16654,basic_helix-loop-helix_protein_lex,G#protein_family_or_group,amino_acid,4,6,6.214608,11.009194,310.072539,1.50,233.5,-0.001611,0.404383
29777,minus_clone_lex,G#cell_line,cell,1,2,7.600902,12.107806,311.074036,2.00,466.0,-0.000643,0.692896
29778,minus_clone_lex,G#cell_line,cell,1,2,7.600902,12.107806,311.074036,2.00,466.0,-0.000643,0.692896
32036,octamer_motif_lex,G#DNA_domain_or_region,nucleotide,8,18,5.521461,9.910582,inf,2.25,449.5,-0.004192,0.808753
32037,octamer_motif_lex,G#DNA_domain_or_region,nucleotide,8,18,5.521461,9.910582,inf,2.25,449.5,-0.004192,0.808753


Term dispersion scores:


,term,sem,class,bi,ni,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
0,'aged'_lymphocyte_lex,G#cell_type,cell,1,1,7.600902,12.800954,0.701595,1.0,244.0,-0.000673,-0.000251
1,'converted'_TCEd_motif_lex,G#DNA_domain_or_region,nucleotide,1,1,7.600902,12.800954,0.701595,1.0,195.0,-0.000538,-0.000251
2,'latency_I'_lex,G#other_name,other,1,1,7.600902,12.800954,0.701595,1.0,206.0,-0.000568,-0.000251
3,'latency_II'_lex,G#other_name,other,1,1,7.600902,12.800954,0.701595,1.0,206.0,-0.000568,-0.000251
4,'master_regulator_lex,G#other_name,other,1,1,7.600902,12.800954,0.701595,1.0,63.0,-0.000174,-0.000251
...,...,...,...,...,...,...,...,...,...,...,...,...
40802,zymogen_plasma_factor_X_lex,G#protein_family_or_group,amino_acid,1,1,7.600902,12.800954,0.701595,1.0,355.0,-0.000979,-0.000251
40803,zymogen_plasma_factors_VII_lex,G#protein_family_or_group,amino_acid,1,1,7.600902,12.800954,0.701595,1.0,355.0,-0.000979,-0.000251
40804,zymography_lex,G#other_name,other,1,1,7.600902,12.800954,0.701595,1.0,191.0,-0.000527,-0.000251
40805,zymosan-treated_cell_lex,G#cell_type,cell,1,1,7.600902,12.800954,0.701595,1.0,225.0,-0.000621,-0.000251


## Compile GENIA Corpus Summary Statistics

This is the result of Table 3 from the manuscript.

In [8]:
# Desginated ordering for the high-level semantic classes
high_level_semantic_class_ord = ['amino_acid', 'nucleotide', 'multi_cell', 'cell', 'other']

# Count number of semantic subclasses in each high-level class
subclass_counts = [len(amino_acid_sems), len(nucleotide_sems), len(multi_cell_sems), len(cell_sems), len(other_sems)]

# Count number of distinct lexical units in each high-level semantic class
lex_unit_counts = term_scores_aug_df.dropna(subset=['class']).groupby('class')['term'].nunique().reindex(high_level_semantic_class_ord).to_list()

# Count number of annotations associated with each high-level semantic class
annotation_counts = term_scores_aug_df.dropna(subset=['class']).groupby('class')['ni'].sum().reindex(high_level_semantic_class_ord).to_list()

# Count number of singletons associated with each high-level semantic class
singleton_counts = term_scores_aug_df[term_scores_aug_df['ni'] == 1].dropna(subset=['class']).groupby('class')['ni'].sum().reindex(high_level_semantic_class_ord).to_list()

# Initialize GENIA summary statistics data frame
genia_summary_stats_df = pd.DataFrame({
    'Semantic class': high_level_semantic_class_ord,
    'Sub-class': subclass_counts,
    'Unique terms': lex_unit_counts,
    'Annotations': annotation_counts,
    'Singletons': singleton_counts
})

# Print GENIA summary statistics to console
display(genia_summary_stats_df)

# Write to CSV
os.makedirs('table-3', exist_ok=True)
genia_summary_stats_df.to_csv('table-3/semantic-class-stats.csv', index=False)

,Semantic class,Sub-class,Unique terms,Annotations,Singletons
0,amino_acid,15,10155,42478,6571
1,nucleotide,12,5574,11619,4115
2,multi_cell,5,1444,5247,961
3,cell,4,4051,11626,2956
4,other,1,10560,19999,8071


## Terminology Extraction Task Experiment

Here we reproduce the result of Tables 5, 6, 7, A1, A2, and A3 from the manuscript.

In [9]:
# Create a minimal data frame of term dispersion scores
term_scores_df = term_scores_aug_df[term_scores_aug_df['ni'] > 1] # Filter out singletons
term_scores_df = term_scores_df.reset_index(drop=True) # Reinitialize row indices
term_scores_df = term_scores_df.drop(columns=['sem', 'class', 'ni', 'bi'])

# Print to console
display(term_scores_df)

,term,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
0,(+)-pentazocine_lex,7.600902,12.107806,311.074036,2.0,522.0,-0.000720,0.692896
1,-120_lex,7.600902,12.107806,311.074036,2.0,360.0,-0.000496,0.692896
2,-150_bp_lex,7.600902,11.702341,inf,3.0,711.0,-0.000654,1.098361
3,-201/-184_NXS_lex,7.600902,12.107806,311.074036,2.0,482.0,-0.000665,0.692896
4,-201_and_-130_lex,7.600902,12.107806,311.074036,2.0,482.0,-0.000665,0.692896
...,...,...,...,...,...,...,...,...
14149,zinc_finger_region_lex,6.907755,12.107806,0.688948,1.0,200.5,-0.001106,-0.000532
14150,zinc_finger_transcription_factor_lex,5.991465,10.855043,122.211280,1.4,201.4,-0.002168,0.335116
14151,zinc_lex,6.907755,10.721512,inf,4.0,447.0,-0.000576,1.385763
14152,zone,6.907755,12.107806,0.688948,1.0,334.5,-0.001845,-0.000532


Define various functions used in the analysis.

In [10]:
# Grab the top k terms
def top_k(dct, k):
  keys = dct.keys()
  values = []
  for key in keys:
    values.append(dct[key][:k])
  keys_values_pair = zip(keys, values)
  return dict(keys_values_pair)

# Count up terms
def count_words(lst, imp_words):
  counter = 0
  for x in lst:
    if x in imp_words:
      counter += 1
  return counter

# Randomly resort term dispersion scores data frame
def resort(term_scores_df):
  sorted_terms = []
  bursty_measure_names = term_scores_df.columns.values.tolist()[1:]

  for measure in bursty_measure_names:
      # Copy the data frame and add a random column
      temp_df = term_scores_df.copy()
      temp_df['random'] = np.random.rand(len(temp_df))
        
      # Sort by the measure and the random column
      sorted_df = temp_df[['term', measure, 'random']].sort_values(by=[measure, 'random'], ascending=[False, True])
        
      # Append the sorted terms to the list
      sorted_terms.append(np.array(sorted_df['term']))
        
      # Drop the random column from the temporary data frame
      temp_df.drop(columns='random', inplace=True)
    
  sorted_terms = np.array(sorted_terms)
  measure_term_pair = zip(bursty_measure_names, sorted_terms)
  sorted_measures = dict(measure_term_pair)
    
  return sorted_measures
    
# Calculate Precision at k scores
def calc_pk(lex_units, sorted_measures, k_values):
  measures = sorted_measures.keys()
  counts = [[], [], [], [], [], [], [], [], [], []]
  pk_dct = dict(zip(measures, counts))  
  for measure in pk_dct.keys():
      for k in k_values:
          pk_dct[measure].append(count_words(top_k(sorted_measures, k)[measure], lex_units)/k)
  result = pd.DataFrame(pk_dct, index=k_values)
  return result

# Calculate Recall at k scores
def calc_rk(lex_units, sorted_measures, k_values):
  measures = sorted_measures.keys()
  counts = [[], [], [], [], [], [], [], [], [], []]
  rk_dct = dict(zip(measures, counts))  
  for measure in rk_dct.keys():
      for k in k_values:
          rk_dct[measure].append(count_words(top_k(sorted_measures, k)[measure], lex_units)/len(lex_units))
  result = pd.DataFrame(rk_dct, index=k_values)
  return result

# Calculate F1 at k scores
def calc_fk(pk, rk):
  result = 2 * (pk * rk) / (pk + rk)
  result = result.fillna(0) # Nan scores are redefined as 0
  return result

# Calculate Rank Biased Overlap scores
def calc_rbo(sorted_measures, k_values):
  RICF = sorted_measures["RICF"]
  measures = sorted_measures.keys()
  counts = [[], [], [], [], [], [], [], [], [], []]
  rbo_dct = dict(zip(measures, counts))  
  for measure in rbo_dct.keys():
      for k in k_values:
          S = top_k(sorted_measures, k)[measure]
          T = RICF[0:k]
          rbo_dct[measure].append(rbo.RankingSimilarity(S, T).rbo())
  result = pd.DataFrame(rbo_dct, index=k_values)
  return result

# Calculate Rank Biased Overlap scores for each semantic class
def calc_rbo2(sorted_measures, categories):
  measures = sorted_measures.keys()
  counts = [[], [], [], [], [], [], [], [], [], []]
  rbo_dct = dict(zip(measures, counts))
  #rbo_dct = dict.fromkeys(measures, [])

  for measure in measures:
      l1 = sorted_measures[measure].tolist()
      
      for category, lex_units in categories.items():
          S = sorted(set(l1) & set(lex_units), key = l1.index)
          l2 = sorted_measures["RICF"].tolist()
          RICF = sorted(set(l2) & set(lex_units), key = l2.index)          
          rbo_dct[measure].append(rbo.RankingSimilarity(S, RICF).rbo()) 

  result = pd.DataFrame(rbo_dct, index=categories.keys())
  return result
    
# Calculate mean P@k, R@k, and F1@k scores
def calc_score_means(nested_list):
    result = []
    num_outer = len(nested_list)
    num_inner = len(nested_list[0])

    for i in range(num_inner):
        means = {}
        for column in nested_list[0][i].columns:
            values = [nested_list[outer][i][column].values for outer in range(num_outer)]
            mean_values = np.mean(values, axis=0)
            means[column] = mean_values
        result.append(pd.DataFrame(means, index=nested_list[0][i].index))
    return result

# Calculate standard deviations of P@k, R@k, and F1@k scores
def calc_score_sds(nested_list):
    result = []
    num_outer = len(nested_list)
    num_inner = len(nested_list[0])

    for i in range(num_inner):
        std_devs = {}
        for column in nested_list[0][i].columns:
            values = [nested_list[outer][i][column].values for outer in range(num_outer)]
            std_values = np.std(values, axis=0, ddof=1)
            std_devs[column] = std_values
        result.append(pd.DataFrame(std_devs, index=nested_list[0][i].index))
    return result

# Calculate mean RBO scores
def calc_mean_rbo_scores(scores_list):
  R = len(scores_list) # Number of replicates
  H = len(scores_list[0]) # Number of dispersion metrics
  d_metrics = scores_list[0].columns # Dispersion metrics by name
  k_values = scores_list[0].index # Top k values
  result = {}
    
  for d_metric in d_metrics:
    scores = [scores_list[r][d_metric].values for r in range(R)]
    mean_scores = np.mean(scores, axis=0)
    result[d_metric] = mean_scores
        
  return pd.DataFrame(result, index=k_values)

Evaluate Precision at k, Recall at k, F1 at k, and RBO scores.

In [11]:
# Initialize a random seed to ensure results can be replicated
np.random.seed(641369)

# Set number of replicates
R = 100 # For testing purposes; change back to 100 for final results

# These are the Precision @ k, Recall @ k and Rank Biased Overlap scores
all_pk_scores = []
all_rk_scores = []
all_fk_scores = []
all_rbo_scores = []
all_rbo_scores2 = []

# Used as inputs for calculating the various scores
k_values = np.array([10, 50, 100, 500, 1000, 5000])
amino_acid = get_high_level_semantic_class_words(amino_acid_sems)
nucleotide = get_high_level_semantic_class_words(nucleotide_sems)
multi_cell = get_high_level_semantic_class_words(multi_cell_sems)
cell = get_high_level_semantic_class_words(cell_sems)
other = get_high_level_semantic_class_words(other_sems)
categories = {
    'all': genia_lexical_units,
    'amino_acid': amino_acid,
    'nucleotide': nucleotide,
    'multi_cell': multi_cell,
    'cell': cell,
    'other': other}

# Calculate evaluation metrics
for r in tqdm(range(R)):
    print('r =', r)
    pk_scores = []
    rk_scores = []
    fk_scores = []
    sorted_measures = resort(term_scores_df)
    
    for category, lex_units in categories.items():
        pk = calc_pk(lex_units, sorted_measures, k_values)
        pk_scores.append(pk)
        rk = calc_rk(lex_units, sorted_measures, k_values)
        rk_scores.append(rk)
        fk = calc_fk(pk, rk)
        fk_scores.append(fk)
    
    all_pk_scores.append(pk_scores)
    all_rk_scores.append(rk_scores)
    all_fk_scores.append(fk_scores)
    rbo_scores = calc_rbo(sorted_measures, k_values)
    all_rbo_scores.append(rbo_scores)
    rbo_scores2 = calc_rbo2(sorted_measures, categories)
    all_rbo_scores2.append(rbo_scores2)

  0%|                                                                                              | 0/100 [00:00<?, ?it/s]

r = 0


  1%|▊                                                                                   | 1/100 [00:44<1:13:49, 44.74s/it]

r = 1


  2%|█▋                                                                                  | 2/100 [01:30<1:14:18, 45.49s/it]

...

 98%|█████████████████████████████████████████████████████████████████████████████████▎ | 98/100 [1:13:48<01:30, 45.04s/it]

r = 98


 99%|██████████████████████████████████████████████████████████████████████████████████▏| 99/100 [1:14:32<00:44, 44.93s/it]

r = 99


100%|██████████████████████████████████████████████████████████████████████████████████| 100/100 [1:15:17<00:00, 45.18s/it]


Save evaluation metrics as Pkl files.

In [12]:
# Ensure the directory exists
os.makedirs('scores-dump', exist_ok=True)

# Write P@k scores to Pkl
with open('scores-dump/all_pk_scores.pkl', 'wb') as file:
    pickle.dump(all_pk_scores, file)

# Write R@k scores to Pkl
with open('scores-dump/all_rk_scores.pkl', 'wb') as file:
    pickle.dump(all_rk_scores, file)

# Write F1@k scores to Pkl
with open('scores-dump/all_fk_scores.pkl', 'wb') as file:
    pickle.dump(all_fk_scores, file)

# Write RBO scores to Pkl
with open('scores-dump/all_rbo_scores.pkl', 'wb') as file:
    pickle.dump(all_rbo_scores, file)

# Write RBO scores as calculated for each semantic class to Pkl
with open('scores-dump/all_rbo_scores2.pkl', 'wb') as file:
    pickle.dump(all_rbo_scores2, file)

In [13]:
# Calculate mean P@k scores and write to CSV
all_pk_scores_means = calc_score_means(all_pk_scores)
os.makedirs('table-5', exist_ok=True)
pd.DataFrame(all_pk_scores_means[0]).to_csv('table-5/all-pk-means.csv', index=False)
pd.DataFrame(all_pk_scores_means[1]).to_csv('table-5/amino_acid-pk-means.csv', index=False)
pd.DataFrame(all_pk_scores_means[2]).to_csv('table-5/nucleotide-pk-means.csv', index=False)
pd.DataFrame(all_pk_scores_means[3]).to_csv('table-5/multicell-pk-means.csv', index=False)
pd.DataFrame(all_pk_scores_means[4]).to_csv('table-5/cell-pk-means.csv', index=False)
pd.DataFrame(all_pk_scores_means[5]).to_csv('table-5/other-pk-means.csv', index=False)

# Calculate standard deviations for P@k scores and write to CSV
all_pk_scores_sds = calc_score_sds(all_pk_scores)
os.makedirs('table-a1', exist_ok=True)
pd.DataFrame(all_pk_scores_sds[0]).to_csv('table-a1/all-pk-sds.csv', index=False)
pd.DataFrame(all_pk_scores_sds[1]).to_csv('table-a1/amino_acid-pk-sds.csv', index=False)
pd.DataFrame(all_pk_scores_sds[2]).to_csv('table-a1/nucleotide-pk-sds.csv', index=False)
pd.DataFrame(all_pk_scores_sds[3]).to_csv('table-a1/multicell-pk-sds.csv', index=False)
pd.DataFrame(all_pk_scores_sds[4]).to_csv('table-a1/cell-pk-sds.csv', index=False)
pd.DataFrame(all_pk_scores_sds[5]).to_csv('table-a1/other-pk-sds.csv', index=False)

In [14]:
# Display mean P@k scores and console
print("Mean P@k scores:")
with pd.option_context('display.precision', 4):
    display(all_pk_scores_means[0].round(4))
    display(all_pk_scores_means[1].round(4))
    display(all_pk_scores_means[2].round(4))
    display(all_pk_scores_means[3].round(4))
    display(all_pk_scores_means[4].round(4))
    display(all_pk_scores_means[5].round(4))

Mean P@k scores:


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
10,0.9290,0.7690,0.9490,1.0000,1.0000,1.0000,1.0000
50,0.9308,0.7670,0.9530,0.9600,1.0000,0.9600,1.0000
100,0.9316,0.7679,0.9524,0.9800,0.9800,0.9400,1.0000
500,0.9277,0.7670,0.9523,0.9835,0.9740,0.9560,0.9913
1000,0.9279,0.7678,0.9529,0.9817,0.9634,0.9514,0.9855
5000,0.8798,0.7683,0.9150,0.9281,0.8992,0.8882,0.9314


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
10,0.3990,0.2420,0.5270,1.0000,0.8000,0.2000,1.0000
50,0.3872,0.2498,0.5414,0.7542,0.7000,0.4000,0.7908
100,0.3959,0.2485,0.5409,0.8200,0.7451,0.4600,0.8300
500,0.3894,0.2500,0.5384,0.6873,0.6220,0.4331,0.6923
1000,0.3900,0.2484,0.5400,0.6441,0.5900,0.4182,0.6433
5000,0.3576,0.2514,0.4293,0.4285,0.4144,0.3614,0.4293


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
10,0.1730,0.1560,0.1520,0.0000,0.1000,0.2000,0.0000
50,0.1808,0.1322,0.1484,0.1000,0.1200,0.1200,0.1000
100,0.1722,0.1318,0.1487,0.1000,0.1000,0.1300,0.1100
500,0.1694,0.1332,0.1516,0.1419,0.1460,0.1349,0.1420
1000,0.1682,0.1338,0.1521,0.1346,0.1396,0.1580,0.1367
5000,0.1557,0.1327,0.1527,0.1554,0.1514,0.1552,0.1558


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
10,0.0410,0.0310,0.0480,0.0000,0.0000,0.0000,0.0000
50,0.0434,0.0404,0.0460,0.0400,0.0400,0.0200,0.0400
100,0.0441,0.0410,0.0433,0.0200,0.0200,0.0200,0.0200
500,0.0437,0.0411,0.0406,0.0333,0.0320,0.0660,0.0334
1000,0.0452,0.0406,0.0402,0.0368,0.0419,0.0580,0.0358
5000,0.0435,0.0410,0.0437,0.0438,0.0410,0.0454,0.0441


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
10,0.1070,0.0940,0.0760,0.0000,0.000,0.1000,0.0000
50,0.1106,0.0898,0.0834,0.0058,0.000,0.0800,0.0092
100,0.1089,0.0938,0.0829,0.0100,0.020,0.0600,0.0100
500,0.1087,0.0942,0.0826,0.0392,0.060,0.0880,0.0400
1000,0.1071,0.0942,0.0816,0.0598,0.069,0.0910,0.0602
5000,0.1009,0.0949,0.0955,0.1001,0.101,0.0994,0.1007


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
10,0.2090,0.2460,0.1460,0.0000,0.1000,0.5000,0.0000
50,0.2088,0.2548,0.1338,0.0600,0.1400,0.3400,0.0600
100,0.2105,0.2528,0.1366,0.0300,0.0949,0.2700,0.0300
500,0.2166,0.2484,0.1391,0.0818,0.1140,0.2340,0.0836
1000,0.2174,0.2508,0.1390,0.1064,0.1230,0.2263,0.1095
5000,0.2222,0.2483,0.1939,0.2002,0.1913,0.2268,0.2016


In [15]:
# Displaye standard deviation of P@k scores to console
print("P@k scores standard deviations:")
with pd.option_context('display.precision', 4):
    display(all_pk_scores_sds[0].round(4))
    display(all_pk_scores_sds[1].round(4))
    display(all_pk_scores_sds[2].round(4))
    display(all_pk_scores_sds[3].round(4))
    display(all_pk_scores_sds[4].round(4))
    display(all_pk_scores_sds[5].round(4))

P@k scores standard deviations:


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
10,0.0832,0.1285,0.0689,0.0000,0.0000,0.0000,0.0000
50,0.0330,0.0584,0.0283,0.0000,0.0000,0.0000,0.0000
100,0.0232,0.0370,0.0217,0.0000,0.0000,0.0000,0.0000
500,0.0112,0.0174,0.0082,0.0010,0.0000,0.0000,0.0011
1000,0.0062,0.0096,0.0052,0.0016,0.0009,0.0005,0.0013
5000,0.0023,0.0003,0.0006,0.0007,0.0000,0.0000,0.0004


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
10,0.1738,0.1350,0.1483,0.0000,0.0000,0.0000,0.0000
50,0.0652,0.0530,0.0654,0.0091,0.0000,0.0000,0.0100
100,0.0483,0.0388,0.0515,0.0000,0.0050,0.0000,0.0000
500,0.0223,0.0168,0.0188,0.0035,0.0000,0.0010,0.0039
1000,0.0136,0.0115,0.0137,0.0043,0.0007,0.0006,0.0039
5000,0.0024,0.0004,0.0011,0.0012,0.0001,0.0000,0.0006


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
10,0.1171,0.1104,0.1078,0.0000,0.0000,0.000,0.0000
50,0.0550,0.0536,0.0520,0.0000,0.0000,0.000,0.0000
100,0.0406,0.0346,0.0356,0.0000,0.0000,0.000,0.0000
500,0.0180,0.0132,0.0155,0.0025,0.0000,0.001,0.0025
1000,0.0109,0.0090,0.0102,0.0036,0.0005,0.000,0.0028
5000,0.0020,0.0003,0.0009,0.0010,0.0001,0.000,0.0003


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
10,0.0605,0.0526,0.0643,0.0000,0.0000,0.0,0.0000
50,0.0268,0.0254,0.0299,0.0000,0.0000,0.0,0.0000
100,0.0218,0.0190,0.0185,0.0000,0.0000,0.0,0.0000
500,0.0082,0.0078,0.0088,0.0016,0.0000,0.0,0.0015
1000,0.0054,0.0053,0.0046,0.0020,0.0007,0.0,0.0013
5000,0.0011,0.0002,0.0005,0.0005,0.0000,0.0,0.0003


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
10,0.0977,0.0851,0.0806,0.0000,0.0000,0.0,0.0000
50,0.0421,0.0372,0.0354,0.0091,0.0000,0.0,0.0100
100,0.0288,0.0280,0.0254,0.0000,0.0000,0.0,0.0000
500,0.0145,0.0114,0.0097,0.0023,0.0000,0.0,0.0026
1000,0.0076,0.0077,0.0067,0.0027,0.0006,0.0,0.0017
5000,0.0013,0.0002,0.0007,0.0009,0.0001,0.0,0.0003


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
10,0.1296,0.1388,0.1150,0.0000,0.0000,0.0000,0.0000
50,0.0493,0.0625,0.0406,0.0000,0.0000,0.0000,0.0000
100,0.0347,0.0442,0.0295,0.0000,0.0050,0.0000,0.0000
500,0.0170,0.0162,0.0114,0.0024,0.0000,0.0000,0.0030
1000,0.0095,0.0105,0.0080,0.0031,0.0007,0.0007,0.0027
5000,0.0025,0.0003,0.0010,0.0012,0.0001,0.0000,0.0005


In [16]:
# Calculate mean R@k scores and write to CSV
all_rk_scores_means = calc_score_means(all_rk_scores)
os.makedirs('table-6', exist_ok=True)
pd.DataFrame(all_rk_scores_means[0]).to_csv('table-6/all-rk-means.csv', index=False)
pd.DataFrame(all_rk_scores_means[1]).to_csv('table-6/amino_acid-rk-means.csv', index=False)
pd.DataFrame(all_rk_scores_means[2]).to_csv('table-6/nucleotide-rk-means.csv', index=False)
pd.DataFrame(all_rk_scores_means[3]).to_csv('table-6/multicell-rk-means.csv', index=False)
pd.DataFrame(all_rk_scores_means[4]).to_csv('table-6/cell-rk-means.csv', index=False)
pd.DataFrame(all_rk_scores_means[5]).to_csv('table-6/other-rk-means.csv', index=False)

# Calculate standard deviations for R@k scores and write to CSV
all_rk_scores_sds = calc_score_sds(all_rk_scores)
os.makedirs('table-a2', exist_ok=True)
pd.DataFrame(all_rk_scores_sds[0]).to_csv('table-a2/all-rk-sds.csv', index=False)
pd.DataFrame(all_rk_scores_sds[1]).to_csv('table-a2/amino_acid-rk-sds.csv', index=False)
pd.DataFrame(all_rk_scores_sds[2]).to_csv('table-a2/nucleotide-rk-sds.csv', index=False)
pd.DataFrame(all_rk_scores_sds[3]).to_csv('table-a2/multicell-rk-sds.csv', index=False)
pd.DataFrame(all_rk_scores_sds[4]).to_csv('table-a2/cell-rk-sds.csv', index=False)
pd.DataFrame(all_rk_scores_sds[5]).to_csv('table-a2/other-rk-sds.csv', index=False)

In [17]:
# Display mean R@k scores console
print("Mean R@k scores:")
with pd.option_context('display.precision', 4):
    display(all_rk_scores_means[0].round(4))
    display(all_rk_scores_means[1].round(4))
    display(all_rk_scores_means[2].round(4))
    display(all_rk_scores_means[3].round(4))
    display(all_rk_scores_means[4].round(4))
    display(all_rk_scores_means[5].round(4))

Mean R@k scores:


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
10,0.0003,0.0002,0.0003,0.0003,0.0003,0.0003,0.0003
50,0.0015,0.0012,0.0015,0.0015,0.0016,0.0015,0.0016
100,0.0029,0.0024,0.0030,0.0031,0.0031,0.0030,0.0031
500,0.0146,0.0121,0.0150,0.0155,0.0153,0.0150,0.0156
1000,0.0292,0.0242,0.0300,0.0309,0.0303,0.0299,0.0310
5000,0.1384,0.1208,0.1439,0.1460,0.1414,0.1397,0.1465


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
10,0.0004,0.0002,0.0005,0.0010,0.0008,0.0002,0.0010
50,0.0019,0.0012,0.0027,0.0037,0.0034,0.0020,0.0039
100,0.0039,0.0024,0.0053,0.0081,0.0073,0.0045,0.0082
500,0.0192,0.0123,0.0265,0.0338,0.0306,0.0213,0.0341
1000,0.0384,0.0245,0.0532,0.0634,0.0581,0.0412,0.0633
5000,0.1761,0.1238,0.2114,0.2110,0.2041,0.1779,0.2114


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
10,0.0003,0.0003,0.0003,0.0000,0.0002,0.0004,0.0000
50,0.0016,0.0012,0.0013,0.0009,0.0011,0.0011,0.0009
100,0.0031,0.0024,0.0027,0.0018,0.0018,0.0023,0.0020
500,0.0152,0.0120,0.0136,0.0127,0.0131,0.0121,0.0127
1000,0.0302,0.0240,0.0273,0.0241,0.0250,0.0283,0.0245
5000,0.1397,0.1190,0.1369,0.1394,0.1358,0.1392,0.1398


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
10,0.0003,0.0002,0.0003,0.0000,0.0000,0.0000,0.0000
50,0.0015,0.0014,0.0016,0.0014,0.0014,0.0007,0.0014
100,0.0031,0.0028,0.0030,0.0014,0.0014,0.0014,0.0014
500,0.0151,0.0142,0.0141,0.0115,0.0111,0.0229,0.0116
1000,0.0313,0.0281,0.0278,0.0255,0.0290,0.0402,0.0248
5000,0.1507,0.1421,0.1513,0.1517,0.1420,0.1572,0.1528


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
10,0.0003,0.0002,0.0002,0.0000,0.0000,0.0002,0.0000
50,0.0014,0.0011,0.0010,0.0001,0.0000,0.0010,0.0001
100,0.0027,0.0023,0.0020,0.0002,0.0005,0.0015,0.0002
500,0.0134,0.0116,0.0102,0.0048,0.0074,0.0109,0.0049
1000,0.0264,0.0233,0.0202,0.0148,0.0170,0.0225,0.0149
5000,0.1245,0.1171,0.1179,0.1236,0.1247,0.1227,0.1242


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
10,0.0002,0.0002,0.0001,0.0000,0.0001,0.0005,0.0000
50,0.0010,0.0012,0.0006,0.0003,0.0007,0.0016,0.0003
100,0.0020,0.0024,0.0013,0.0003,0.0009,0.0026,0.0003
500,0.0103,0.0118,0.0066,0.0039,0.0054,0.0111,0.0040
1000,0.0206,0.0237,0.0132,0.0101,0.0116,0.0214,0.0104
5000,0.1052,0.1175,0.0918,0.0948,0.0906,0.1074,0.0954


In [18]:
# Display standard deviation of R@k scores to console
print("R@k scores standard deviations:")
with pd.option_context('display.precision', 4):
    display(all_rk_scores_sds[0].round(4))
    display(all_rk_scores_sds[1].round(4))
    display(all_rk_scores_sds[2].round(4))
    display(all_rk_scores_sds[3].round(4))
    display(all_rk_scores_sds[4].round(4))
    display(all_rk_scores_sds[5].round(4))

R@k scores standard deviations:


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
10,0.0000,0.0000,0.0000,0.0000,0.0,0.0,0.0000
50,0.0001,0.0001,0.0000,0.0000,0.0,0.0,0.0000
100,0.0001,0.0001,0.0001,0.0000,0.0,0.0,0.0000
500,0.0002,0.0003,0.0001,0.0000,0.0,0.0,0.0000
1000,0.0002,0.0003,0.0002,0.0001,0.0,0.0,0.0000
5000,0.0004,0.0001,0.0001,0.0001,0.0,0.0,0.0001


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
10,0.0002,0.0001,0.0001,0.0000,0.0000,0.0000,0.0000
50,0.0003,0.0003,0.0003,0.0000,0.0000,0.0000,0.0000
100,0.0005,0.0004,0.0005,0.0000,0.0000,0.0000,0.0000
500,0.0011,0.0008,0.0009,0.0002,0.0000,0.0000,0.0002
1000,0.0013,0.0011,0.0013,0.0004,0.0001,0.0001,0.0004
5000,0.0012,0.0002,0.0005,0.0006,0.0000,0.0000,0.0003


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
10,0.0002,0.0002,0.0002,0.0000,0.0000,0.0000,0.0000
50,0.0005,0.0005,0.0005,0.0000,0.0000,0.0000,0.0000
100,0.0007,0.0006,0.0006,0.0000,0.0000,0.0000,0.0000
500,0.0016,0.0012,0.0014,0.0002,0.0000,0.0001,0.0002
1000,0.0020,0.0016,0.0018,0.0006,0.0001,0.0000,0.0005
5000,0.0018,0.0003,0.0008,0.0009,0.0001,0.0000,0.0003


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
10,0.0004,0.0004,0.0004,0.0000,0.0000,0.0,0.0000
50,0.0009,0.0009,0.0010,0.0000,0.0000,0.0,0.0000
100,0.0015,0.0013,0.0013,0.0000,0.0000,0.0,0.0000
500,0.0028,0.0027,0.0030,0.0006,0.0000,0.0,0.0005
1000,0.0037,0.0036,0.0032,0.0014,0.0005,0.0,0.0009
5000,0.0037,0.0006,0.0018,0.0019,0.0001,0.0,0.0010


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
10,0.0002,0.0002,0.0002,0.0000,0.0000,0.0,0.0000
50,0.0005,0.0005,0.0004,0.0001,0.0000,0.0,0.0001
100,0.0007,0.0007,0.0006,0.0000,0.0000,0.0,0.0000
500,0.0018,0.0014,0.0012,0.0003,0.0000,0.0,0.0003
1000,0.0019,0.0019,0.0017,0.0007,0.0002,0.0,0.0004
5000,0.0016,0.0003,0.0009,0.0011,0.0001,0.0,0.0004


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
10,0.0001,0.0001,0.0001,0.0000,0.0000,0.0000,0.0000
50,0.0002,0.0003,0.0002,0.0000,0.0000,0.0000,0.0000
100,0.0003,0.0004,0.0003,0.0000,0.0000,0.0000,0.0000
500,0.0008,0.0008,0.0005,0.0001,0.0000,0.0000,0.0001
1000,0.0009,0.0010,0.0008,0.0003,0.0001,0.0001,0.0003
5000,0.0012,0.0002,0.0005,0.0006,0.0000,0.0000,0.0003


In [19]:
# Calculate mean F1@k scores and write to CSV
all_fk_scores_means = calc_score_means(all_fk_scores)
os.makedirs('table-7', exist_ok=True)
pd.DataFrame(all_fk_scores_means[0]).to_csv('table-7/all-fk-means.csv', index=False)
pd.DataFrame(all_fk_scores_means[1]).to_csv('table-7/amino_acid-fk-means.csv', index=False)
pd.DataFrame(all_fk_scores_means[2]).to_csv('table-7/nucleotide-fk-means.csv', index=False)
pd.DataFrame(all_fk_scores_means[3]).to_csv('table-7/multicell-fk-means.csv', index=False)
pd.DataFrame(all_fk_scores_means[4]).to_csv('table-7/cell-fk-means.csv', index=False)
pd.DataFrame(all_fk_scores_means[5]).to_csv('table-7/other-fk-means.csv', index=False)

# Calculate standard deviations for F1@k scores and write to CSV
all_fk_scores_sds = calc_score_sds(all_fk_scores)
os.makedirs('table-a3', exist_ok=True)
pd.DataFrame(all_fk_scores_sds[0]).to_csv('table-a3/all-fk-sds.csv', index=False)
pd.DataFrame(all_fk_scores_sds[1]).to_csv('table-a3/amino_acid-fk-sds.csv', index=False)
pd.DataFrame(all_fk_scores_sds[2]).to_csv('table-a3/nucleotide-fk-sds.csv', index=False)
pd.DataFrame(all_fk_scores_sds[3]).to_csv('table-a3/multicell-fk-sds.csv', index=False)
pd.DataFrame(all_fk_scores_sds[4]).to_csv('table-a3/cell-fk-sds.csv', index=False)
pd.DataFrame(all_fk_scores_sds[5]).to_csv('table-a3/other-fk-sds.csv', index=False)

In [20]:
# Display mean F1@k scores to console
print("Mean F1@k scores:")
with pd.option_context('display.precision', 4):
    display(all_fk_scores_means[0].round(4))
    display(all_fk_scores_means[1].round(4))
    display(all_fk_scores_means[2].round(4))
    display(all_fk_scores_means[3].round(4))
    display(all_fk_scores_means[4].round(4))
    display(all_fk_scores_means[5].round(4))

Mean F1@k scores:


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
10,0.0006,0.0005,0.0006,0.0006,0.0006,0.0006,0.0006
50,0.0029,0.0024,0.0030,0.0030,0.0031,0.0030,0.0031
100,0.0058,0.0048,0.0060,0.0061,0.0061,0.0059,0.0063
500,0.0287,0.0238,0.0295,0.0305,0.0302,0.0296,0.0307
1000,0.0566,0.0468,0.0581,0.0599,0.0588,0.0580,0.0601
5000,0.2392,0.2088,0.2487,0.2523,0.2444,0.2414,0.2532


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
10,0.0008,0.0005,0.0010,0.0020,0.0016,0.0004,0.0020
50,0.0038,0.0024,0.0053,0.0074,0.0069,0.0039,0.0077
100,0.0077,0.0048,0.0105,0.0160,0.0145,0.0090,0.0162
500,0.0365,0.0235,0.0505,0.0645,0.0584,0.0406,0.0650
1000,0.0699,0.0445,0.0968,0.1155,0.1058,0.0750,0.1153
5000,0.2359,0.1659,0.2833,0.2828,0.2735,0.2385,0.2833


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
10,0.0006,0.0006,0.0005,0.0000,0.0004,0.0007,0.0000
50,0.0032,0.0024,0.0026,0.0018,0.0021,0.0021,0.0018
100,0.0061,0.0046,0.0052,0.0035,0.0035,0.0046,0.0039
500,0.0279,0.0219,0.0250,0.0234,0.0240,0.0222,0.0234
1000,0.0512,0.0407,0.0463,0.0409,0.0425,0.0481,0.0416
5000,0.1473,0.1255,0.1444,0.1469,0.1432,0.1468,0.1473


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
10,0.0006,0.0004,0.0007,0.0000,0.0000,0.0000,0.0000
50,0.0029,0.0027,0.0031,0.0027,0.0027,0.0013,0.0027
100,0.0057,0.0053,0.0056,0.0026,0.0026,0.0026,0.0026
500,0.0225,0.0211,0.0209,0.0172,0.0165,0.0340,0.0172
1000,0.0369,0.0333,0.0329,0.0301,0.0343,0.0475,0.0293
5000,0.0675,0.0637,0.0678,0.0680,0.0636,0.0705,0.0685


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
10,0.0005,0.0005,0.0004,0.0000,0.0000,0.0005,0.0000
50,0.0027,0.0022,0.0020,0.0001,0.0000,0.0020,0.0002
100,0.0052,0.0045,0.0040,0.0005,0.0010,0.0029,0.0005
500,0.0239,0.0207,0.0182,0.0086,0.0132,0.0193,0.0088
1000,0.0424,0.0373,0.0323,0.0237,0.0273,0.0360,0.0238
5000,0.1115,0.1048,0.1055,0.1106,0.1116,0.1098,0.1112


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
10,0.0004,0.0005,0.0003,0.0000,0.0002,0.0009,0.0000
50,0.0020,0.0024,0.0013,0.0006,0.0013,0.0032,0.0006
100,0.0039,0.0047,0.0026,0.0006,0.0018,0.0051,0.0006
500,0.0196,0.0225,0.0126,0.0074,0.0103,0.0212,0.0076
1000,0.0376,0.0434,0.0240,0.0184,0.0213,0.0391,0.0190
5000,0.1428,0.1595,0.1246,0.1287,0.1229,0.1458,0.1295


In [21]:
# Display standard deviation of F1@k scores to console
print("F1@k scores standard deviations:")
with pd.option_context('display.precision', 4):
    display(all_fk_scores_sds[0].round(4))
    display(all_fk_scores_sds[1].round(4))
    display(all_fk_scores_sds[2].round(4))
    display(all_fk_scores_sds[3].round(4))
    display(all_fk_scores_sds[4].round(4))
    display(all_fk_scores_sds[5].round(4))

F1@k scores standard deviations:


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
10,0.0001,0.0001,0.0000,0.0000,0.0000,0.0,0.0000
50,0.0001,0.0002,0.0001,0.0000,0.0000,0.0,0.0000
100,0.0001,0.0002,0.0001,0.0000,0.0000,0.0,0.0000
500,0.0003,0.0005,0.0003,0.0000,0.0000,0.0,0.0000
1000,0.0004,0.0006,0.0003,0.0001,0.0001,0.0,0.0001
5000,0.0006,0.0001,0.0002,0.0002,0.0000,0.0,0.0001


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
10,0.0003,0.0003,0.0003,0.0000,0.0000,0.0000,0.0000
50,0.0006,0.0005,0.0006,0.0001,0.0000,0.0000,0.0001
100,0.0009,0.0008,0.0010,0.0000,0.0001,0.0000,0.0000
500,0.0021,0.0016,0.0018,0.0003,0.0000,0.0001,0.0004
1000,0.0024,0.0021,0.0025,0.0008,0.0001,0.0001,0.0007
5000,0.0016,0.0002,0.0007,0.0008,0.0001,0.0000,0.0004


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
10,0.0004,0.0004,0.0004,0.0000,0.0000,0.0000,0.0000
50,0.0010,0.0010,0.0009,0.0000,0.0000,0.0000,0.0000
100,0.0014,0.0012,0.0013,0.0000,0.0000,0.0000,0.0000
500,0.0030,0.0022,0.0025,0.0004,0.0000,0.0002,0.0004
1000,0.0033,0.0027,0.0031,0.0011,0.0002,0.0000,0.0008
5000,0.0019,0.0003,0.0008,0.0009,0.0001,0.0000,0.0003


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
10,0.0008,0.0007,0.0009,0.0000,0.0000,0.0,0.0000
50,0.0018,0.0017,0.0020,0.0000,0.0000,0.0,0.0000
100,0.0028,0.0025,0.0024,0.0000,0.0000,0.0,0.0000
500,0.0042,0.0040,0.0045,0.0008,0.0000,0.0,0.0008
1000,0.0044,0.0043,0.0037,0.0016,0.0006,0.0,0.0011
5000,0.0017,0.0003,0.0008,0.0008,0.0001,0.0,0.0004


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
10,0.0005,0.0004,0.0004,0.0000,0.0000,0.0,0.0000
50,0.0010,0.0009,0.0009,0.0002,0.0000,0.0,0.0002
100,0.0014,0.0013,0.0012,0.0000,0.0000,0.0,0.0000
500,0.0032,0.0025,0.0021,0.0005,0.0000,0.0,0.0006
1000,0.0030,0.0030,0.0027,0.0011,0.0002,0.0,0.0007
5000,0.0014,0.0003,0.0008,0.0010,0.0001,0.0,0.0004


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
10,0.0002,0.0003,0.0002,0.0000,0.0000,0.0000,0.0000
50,0.0005,0.0006,0.0004,0.0000,0.0000,0.0000,0.0000
100,0.0007,0.0008,0.0006,0.0000,0.0001,0.0000,0.0000
500,0.0015,0.0015,0.0010,0.0002,0.0000,0.0000,0.0003
1000,0.0016,0.0018,0.0014,0.0005,0.0001,0.0001,0.0005
5000,0.0016,0.0002,0.0006,0.0008,0.0001,0.0000,0.0003


In [22]:
# Calculate mean RBO scores
mean_rbo_scores = calc_mean_rbo_scores(all_rbo_scores)

# Write to CSV
os.makedirs('table-8', exist_ok=True)
pd.DataFrame(mean_rbo_scores).to_csv('table-8/rbo-means.csv', index=False)

# Display RBO scores to console
with pd.option_context('display.precision', 4):
    display(mean_rbo_scores.round(4))

,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
10,0.0006,0.000,0.0011,0.9167,0.6142,0.0000,1.0
50,0.0059,0.000,0.0078,0.9001,0.6860,0.0000,1.0
100,0.0110,0.000,0.0194,0.9093,0.7011,0.0000,1.0
500,0.0530,0.000,0.0987,0.9312,0.7287,0.0147,1.0
1000,0.0993,0.000,0.1974,0.9356,0.7655,0.0561,1.0
5000,0.4518,0.129,0.6530,0.9065,0.8016,0.4215,1.0


In [23]:
# Calculate mean RBO scores by semantic class
mean_rbo_scores2 = calc_mean_rbo_scores(all_rbo_scores2)

# Write to CSV
os.makedirs('table-9', exist_ok=True)
pd.DataFrame(mean_rbo_scores2).to_csv('table-9/rbo-means-by-semantic-class.csv', index=False)

# Display RBO scores to console
with pd.option_context('display.precision', 4):
    display(mean_rbo_scores2.round(4))

,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
all,0.6312,0.4292,0.8133,0.9385,0.8665,0.6154,1.0
amino_acid,0.5972,0.3856,0.7749,0.9564,0.8818,0.5846,1.0
nucleotide,0.6449,0.4328,0.8060,0.9270,0.8645,0.6231,1.0
multi_cell,0.6442,0.4532,0.8304,0.9318,0.8486,0.6198,1.0
cell,0.6577,0.4757,0.8310,0.9201,0.8498,0.6422,1.0
other,0.6797,0.4822,0.8436,0.9050,0.8304,0.6593,1.0


## Top 10 Ranked Terms Example

Here we reproduce the result of Table 10 from the manuscript.

In [24]:
# Initialize a random seed to ensure results can be replicated
np.random.seed(641369)

# Retrieve top 10 ranked terms
top = 10
ranked_terms_df = resort(term_scores_df)
top_10_ranked_terms_df = pd.DataFrame(top_k(ranked_terms_df, top))

# Print to console
display(top_10_ranked_terms_df)

# Write to CSV
os.makedirs('table-10', exist_ok=True)
top_10_ranked_terms_df.to_csv('table-10/top-10-terms.csv', index=False)

,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
0,V14Rho_lex,marginally,TCL1_lex,Bcl-6_lex,Bcl-6_lex,D_variant_cell_lex,Bcl-6_lex
1,lymphoid_progenitor_cell_lex,dispersed,RBP-Jkappa_lex,v-erbA_lex,TCRzeta_lex,combined_pituitary_hormone_deficiency_lex,SMX_lex
2,achievable,campomelic_dysplasia_lex,CD28_lex,SMX_lex,ML-9_lex,Pit-1_gene_lex,v-erbA_lex
3,cytokinestimulated,IL-2_receptor_beta-chain_lex,RAI_lex,SHP1_lex,AITL_lex,EBNA-5_lex,SHP1_lex
4,empirically,tyrosine_kinase_receptor_lex,chromosome_16_lex,ML-9_lex,SHP1_lex,palmar_fibromatosis_lex,ML-9_lex
5,Bcl-x(L)_lex,pancreatic,LMP_promoter_lex,beta-casein_lex,beta-casein_lex,differentiation-inducing_activity_lex,beta-casein_lex
6,SP100_lex,UT-7_Epo_cell_lex,ZEB_lex,I_kappaB_lex,A-myb_lex,Toremifene_lex,I_kappaB_lex
7,bipolar_disorder_lex,caspase_inhibitor_lex,respiratory_epithelium_lex,p95vav_lex,I_kappaB_lex,lymphocyte_death_lex,p95vav_lex
8,PB1_lex,mouse_AML1/PEBP2_alpha_B_lex,ZAP-70_lex,TCRzeta_lex,SMX_lex,DEN_lex,TCRzeta_lex
9,RANTES_gene_lex,TATA_element_lex,SN50_lex,EBNA-2_lex,Rap1_protein_lex,rickets-like_disease_lex,DM_lex


## Stopwords Exploratory Analysis

Here we reproduce the result of Table 11 from the manuscript.

In [25]:
def getrank(sorted_measures):
    unique_terms = set()
    for terms in sorted_measures.values():
        unique_terms.update(terms)
    unique_terms = sorted(unique_terms)
    
    # Create a data frame to hold the rankings
    ranking_df = pd.DataFrame(index=unique_terms, columns=sorted_measures.keys())
    
    # Fill the data frame with rankings
    for measure, terms in sorted_measures.items():
        for rank, term in enumerate(terms):
            ranking_df.at[term, measure] = rank + 1  # Rank starts from 1
    
    # Replace NaN with a large number to indicate unranked terms
    ranking_df = ranking_df.fillna(len(unique_terms) + 1)
    #csv_file_path = 'ranking_table.csv'
    #ranking_df.to_csv(csv_file_path)
    return ranking_df

# Function to filter stopwords from the ranking data frame
def filter_stopwords(ranking_df):
    stopwords_list = set(stopwords.words('english'))
    
    # Filter the data frame to include only stopwords
    stopwords_rank = ranking_df[ranking_df.index.isin(stopwords_list)]
    
    # Save the stopwords ranking data frame to a CSV file
    #csv_file_path = 'stopwords_ranking_table.csv'
    #stopwords_rank.to_csv(csv_file_path)
    
    return stopwords_rank

In [26]:
# Initialize a random seed to ensure results can be replicated
np.random.seed(641369)

# Generate term dispersion ranks for R different versions of the data
all_quantiles_df = []
for r in tqdm(range(R)):
    sorted_measures = resort(term_scores_df)
    rank = getrank(sorted_measures)
    stopwords_ranks_df = filter_stopwords(rank)
    bursty_measure_names = stopwords_ranks_df.head(0)
    quantiles = []
    for bursty_measure_name in bursty_measure_names:
        quantiles.append(stopwords_ranks_df[bursty_measure_name].quantile([0, 0.25, 0.5, 0.75, 1]))
    quantiles_df = pd.DataFrame(quantiles)
    all_quantiles_df.append(quantiles_df)

# Extract the column and index names from the first quantiles data frame
columns = all_quantiles_df[0].columns
index = all_quantiles_df[0].index

# Initialize empty data frames to store the mean and standard deviation values
mean_df = pd.DataFrame(index=index, columns=columns)
std_df = pd.DataFrame(index=index, columns=columns)

# Compute the mean and standard deviation of corresponding elements across all matrices
for col in columns:
    for idx in index:
        values = [matrix.at[idx, col] for matrix in all_quantiles_df]
        mean_df.at[idx, col] = np.mean(values)
        std_df.at[idx, col] = np.std(values)

100%|████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:40<00:00,  2.48it/s]


In [27]:
# Print to console
print("Mean values:")
with pd.option_context('display.precision', 4):
    display(mean_df)
print("\nStandard deviations:")
with pd.option_context('display.precision', 4):
    display(std_df)

# Write to CSV
os.makedirs('table-11', exist_ok=True)
mean_df.to_csv('table-11/stopword-rank-means.csv')
std_df.to_csv('table-11/stopword-rank-sds.csv')

Mean values:


,0.00,0.25,0.50,0.75,1.00
IDF,1916.45,13361.42,13968.545,14122.25,14154.0
ICF,854.59,13270.42,13936.445,14123.25,14154.0
Chi-sq,725.62,7237.0,8053.0,8649.5,14153.0
CG,19.0,5602.5,7690.47,8523.9175,13889.61
ICB,70.0,5375.0,7542.5,8941.25,13910.16
DoP,3232.68,13354.25,13971.0,14123.0,14154.0
RICF,2427.0,7800.6625,8224.5,8775.8825,14153.0



Standard deviations:


,0.00,0.25,0.50,0.75,1.00
IDF,1040.5928,3.3572,0.3542,0.0,0.0
ICF,743.093,2.7879,0.616,0.0,0.0
Chi-sq,550.5873,0.0,0.0,0.0,0.0
CG,0.0,0.0,0.7239,2.0512,233.3185
ICB,0.0,0.0,0.0,0.0,3.2085
DoP,4.0519,0.0,0.0,0.0,0.0
RICF,0.0,0.3731,0.0,0.1248,0.0
